In [ ]:
from pathlib import Path
from dtale import show
import polars as pl
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / ".." / "data"
df = pl.read_parquet(data_dir / "parsed_conversations" / "cm0i27jdj0000aqpa73ghpcxf.snappy")



In [ ]:
df = (
    df.with_columns(
        pl.concat_str(
            [
                pl.col("date"),
                pl.lit(" at "),
                pl.col("time"),
                pl.lit("\n QUESTION: "),
                pl.col("question"),
                pl.lit("\n ANSWER: "),
                pl.col("answer"),
            ],
        ).alias("conversation_text")
    )
    .group_by("conversation_id")
    .agg(
        [
            pl.col("conversation_text").str.concat("\n").alias("conversation_text"),
            pl.col("date").first().alias("start_date"),
            pl.col("time").first().alias("start_time"),
            pl.col("date").last().alias("end_date"),
            pl.col("time").last().alias("end_time"),
            pl.col("question").count().alias("message_count"),
        ]
    )
)


In [ ]:
show(df.to_pandas()).open_browser()

In [ ]:
print(df['conversation_text'][30])

**Prompt 1**

You are tasked with analyzing Q&A conversations and categorizing statements into three levels of certainty: Observable, Inferrable, and Speculative. Your output should be in JSON format, with reasoning presented before statements.

Categories:

Observable: Direct evidence present in the data
- Like reviewing security camera footage - simply stating what's there
- Requires no interpretation, just accurate description
- Any observer would agree on these facts
Example: When someone asks "How do I configure PostgreSQL with Django?", we can observe they asked about Django-PostgreSQL integration - this is simply stating what's in the data


Inferrable: Reasonable conclusions based on the evidence
- Like a detective connecting pieces of evidence
- Requires domain expertise to make logical connections
- Most qualified observers would reach similar conclusions
Example: If someone asks about Django basics, then deployment, then PostgreSQL integration, we can infer they're building a production web application - it's not directly stated but follows logically from the pattern


Speculative: Possible underlying context given available information
- Like a psychologist understanding motivations and life context from behaviors
- Requires understanding of human/organizational behavior
- Different analysts might reach different but valid conclusions
Example: From the same Django/PostgreSQL questions, speculating "They're scaling their side project into a business" - while plausible, this involves more uncertainty

CLASSIFICATION PROCESS:

Read the Q&A conversation carefully and for each question, ask:
- Can I quote text that directly proves this? → Observable
- Can I connect multiple pieces of evidence to support this? → Inferrable
- Am I making assumptions beyond the evidence? → Speculative

IMPORTANT: For the speculatives, be as ambitious and broad as you can even if the results are low confidence.

Output Format (JSON):
[
  {
    "statement": "Statement text",
    "category": "observable" | "inferrable" | "speculative",
    "reasoning": "Reasoning for the category",
    "confidence": 0-100,
  },
  ...
]

Here is the data:


**Prompt 2**

Using the previously categorized statements from the Q&A, extract causal relationships between [S]peculative, [I]nferrable, and [O]bservable elements. Causation flows from Speculative insights through Inferrable patterns to Observable behaviors.

Format: [Category1] Entity1 -> caused -> [Category2] Entity2

Example:
Given categorized statements:
[O] Watching coding videos for months
[O] Asking how to start Python
[O] Wanting to build real projects
[I] Moving from tutorials to practice
[I] Beginning Python journey
[S] Career pressure from team expansion
[S] Competition with younger colleagues

Causal relationships:
[S] career pressure -> caused -> [S] peer competition anxiety
[S] peer competition anxiety -> caused -> [I] tutorial-to-practice transition
[I] tutorial-to-practice transition -> caused -> [O] seeking Python basics
[I] tutorial-to-practice transition -> caused -> [O] wanting real projects
[O] watching tutorials -> caused -> [O] readiness for coding

Key principle: Causation can only flow down (S→I→O) or across (S→S, I→I, O→O), never up.